# Укладка бедра

Image-only и hybrid учатся на одних и тех же пяти фолдах по `study_id`. В инференсе стоит смесь `0.7 * image + 0.3 * hybrid` с порогом `0.193505`. Манифест по AUC выбирает смесь `0.5 / 0.5`.

## Настройки


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torchvision.transforms as T
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

import dxa_utils as U

RANDOM_STATE = 42
IMG_SIZE = 320
EPOCHS = 15
BATCH_SIZE = 8
BACKBONE_LR = 2e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AUGMENTATION = True
AUG_BRIGHTNESS = 0.10
AUG_CONTRAST = 0.10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COL = "femur_lay"
OUTPUT_DIR = Path("femur_lay_sideaware_v5_final")
OUTPUT_DIR.mkdir(exist_ok=True)
NORMALIZE = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
GEOM_NAMES = [
    "height_mm",
    "com_x",
    "com_y",
    "upper_com_x",
    "lower_com_x",
    "upper_lower_dx",
    "shaft_offset_b",
    "bright_mass_ratio",
    "bright_bbox_x0",
    "bright_bbox_x1",
    "bright_bbox_y0",
    "bright_bbox_y1",
]


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
print("DEVICE:", DEVICE)
print("torch:", torch.__version__)


## Данные и фолды

В обучение входят только кадры со стороной L или R и заполненным `femur_lay`. Строки с комментарием «эндопротез» исключаются. Пять фолдов общие для image-only и hybrid, группа — `study_id`.


In [ ]:
df = pd.read_csv("femur_img_labels.csv")
required = {"study_id", "file_path", "femur_side", LABEL_COL, "comment"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

train_df = df[df["femur_side"].isin(["L", "R"]) & df[LABEL_COL].notna()].copy()
prosthesis_mask = train_df["comment"].fillna("").str.contains(
    "эндопротез", case=False, regex=False
)
print(
    "prosthesis rows excluded:",
    int(prosthesis_mask.sum()),
    "studies:",
    train_df.loc[prosthesis_mask, "study_id"].nunique(),
)
train_df = train_df.loc[~prosthesis_mask].reset_index(drop=True)
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
print(
    f"images={len(train_df)}, "
    f"studies={train_df.study_id.nunique()}, "
    f"positives={int(train_df[LABEL_COL].sum())}, "
    f"negatives={int((train_df[LABEL_COL] == 0).sum())}"
)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(
    sgkf.split(train_df, train_df[LABEL_COL], groups=train_df["study_id"])
):
    train_df.loc[val_idx, "fold"] = fold
print(pd.crosstab(train_df["fold"], train_df[LABEL_COL]))
train_df.to_csv(OUTPUT_DIR / "dataset_v5.csv", index=False)


## Кэш снимков

Кадр приводится к стороне из разметки и переводится в uint8. Геометрия считается уже по этому кадру и ещё раз вызывает `canonicalize_femur(..., "R")`.


In [ ]:
def normalize_uint8(arr):
    arr = np.asarray(arr).astype(np.float32)
    lo = float(arr.min())
    hi = float(arr.max())
    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.uint8)
    arr = (arr - lo) / (hi - lo)
    return np.clip(arr * 255.0, 0, 255).astype(np.uint8)


def make_transform(train):
    ops = [T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE))]
    if train and USE_AUGMENTATION:
        ops.append(T.ColorJitter(brightness=AUG_BRIGHTNESS, contrast=AUG_CONTRAST))
    ops.append(T.ToTensor())
    return T.Compose(ops)


def _shifted(arr, pct=20):
    values = np.asarray(arr, dtype=np.float64)
    values = values - np.percentile(values, pct)
    values[values < 0] = 0
    return values


def _weighted_com(block):
    values = np.asarray(block, dtype=np.float64)
    height, width = values.shape
    yy, xx = np.indices((height, width), dtype=np.float64)
    mass = values.sum()
    if mass <= 1e-9:
        return 0.5, 0.5
    cx = float((values * xx).sum() / mass / max(width - 1, 1))
    cy = float((values * yy).sum() / mass / max(height - 1, 1))
    return cx, cy


def geometry_features(arr):
    canonical = U.canonicalize_femur(arr, "R")
    values = _shifted(canonical)
    height, width = values.shape
    com_x, com_y = _weighted_com(values)
    upper = values[: height // 3]
    lower = values[2 * height // 3 :]
    upper_com_x, _ = _weighted_com(upper)
    lower_com_x, _ = _weighted_com(lower)
    shaft_b = U.shaft_offset_b(canonical)
    mask = values >= np.percentile(values, 85)
    ys, xs = np.where(mask)
    if len(xs):
        bx0 = float(xs.min() / max(width - 1, 1))
        bx1 = float(xs.max() / max(width - 1, 1))
        by0 = float(ys.min() / max(height - 1, 1))
        by1 = float(ys.max() / max(height - 1, 1))
    else:
        bx0 = bx1 = by0 = by1 = 0.5
    bright_mass_ratio = float(values[mask].sum() / (values.sum() + 1e-9))
    return np.array(
        [
            height * U.PX_Y_MM,
            com_x,
            com_y,
            upper_com_x,
            lower_com_x,
            upper_com_x - lower_com_x,
            shaft_b,
            bright_mass_ratio,
            bx0,
            bx1,
            by0,
            by1,
        ],
        dtype=np.float32,
    )


cache = {}
for _, row in train_df.iterrows():
    arr = pydicom.dcmread(row["file_path"]).pixel_array
    canonical = normalize_uint8(U.canonicalize_femur(arr, row["femur_side"]))
    cache[row["file_path"]] = {
        "image": canonical,
        "geometry": geometry_features(canonical),
    }
print("Cached images:", len(cache))


## Датасеты и модели


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, frame, train=False):
        self.df = frame.reset_index(drop=True).copy()
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = self.transform(cache[row["file_path"]]["image"])
        x = NORMALIZE(x.repeat(3, 1, 1))
        y = torch.tensor([float(row[LABEL_COL])], dtype=torch.float32)
        return x, y


class HybridDataset(Dataset):
    def __init__(self, frame, train=False):
        self.df = frame.reset_index(drop=True).copy()
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        item = cache[row["file_path"]]
        x = NORMALIZE(self.transform(item["image"]).repeat(3, 1, 1))
        g = torch.tensor(item["geometry"], dtype=torch.float32)
        y = torch.tensor([float(row[LABEL_COL])], dtype=torch.float32)
        return x, g, y


class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        for name, param in net.named_parameters():
            param.requires_grad = name.startswith("layer3") or name.startswith("layer4")
        net.fc = nn.Linear(net.fc.in_features, 1)
        self.net = net

    def forward(self, x):
        return self.net(x)


class HybridModel(nn.Module):
    def __init__(self, n_geom):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        for name, param in backbone.named_parameters():
            param.requires_grad = name.startswith("layer3") or name.startswith("layer4")
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.geom_head = nn.Sequential(
            nn.Linear(n_geom, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, 1),
        )

    def forward(self, x_img, x_geom):
        return self.classifier(torch.cat([self.backbone(x_img), self.geom_head(x_geom)], dim=1))


## Функции обучения


In [ ]:
def pos_weight(frame):
    pos = float(frame[LABEL_COL].sum())
    neg = float(len(frame) - pos)
    return torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=DEVICE)


def setup_optimizer(model):
    backbone = []
    head = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if (
            name.startswith("net.layer3")
            or name.startswith("net.layer4")
            or name.startswith("backbone.layer3")
            or name.startswith("backbone.layer4")
        ):
            backbone.append(param)
        else:
            head.append(param)
    return torch.optim.AdamW(
        [
            {"params": backbone, "lr": BACKBONE_LR},
            {"params": head, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def _run_epoch(model, loader, criterion, optimizer):
    model.train()
    losses = []
    for batch in loader:
        batch = [item.to(DEVICE) for item in batch]
        y = batch[-1]
        inputs = batch[:-1]
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(*inputs), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        losses.append(float(loss.item()))
    return float(np.mean(losses))


def train_image_model(frame):
    set_seed(RANDOM_STATE)
    model = ImageModel().to(DEVICE)
    loader = DataLoader(
        ImageDataset(frame, train=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )
    optimizer = setup_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight(frame))
    history = []
    for epoch in range(EPOCHS):
        history.append({"epoch": epoch + 1, "loss": _run_epoch(model, loader, criterion, optimizer)})
        scheduler.step()
    return model, pd.DataFrame(history)


def train_hybrid_model(frame):
    set_seed(RANDOM_STATE)
    model = HybridModel(len(GEOM_NAMES)).to(DEVICE)
    loader = DataLoader(
        HybridDataset(frame, train=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )
    optimizer = setup_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight(frame))
    history = []
    for epoch in range(EPOCHS):
        history.append({"epoch": epoch + 1, "loss": _run_epoch(model, loader, criterion, optimizer)})
        scheduler.step()
    return model, pd.DataFrame(history)


@torch.no_grad()
def predict_image_model(model, frame):
    model.eval()
    loader = DataLoader(ImageDataset(frame, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    out = []
    for x, _ in loader:
        out.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy().reshape(-1))
    return np.concatenate(out)


@torch.no_grad()
def predict_hybrid_model(model, frame):
    model.eval()
    loader = DataLoader(HybridDataset(frame, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    out = []
    for x_img, x_geom, _ in loader:
        out.append(torch.sigmoid(model(x_img.to(DEVICE), x_geom.to(DEVICE))).cpu().numpy().reshape(-1))
    return np.concatenate(out)


## Обучение по фолдам

In [ ]:
image_oof = np.full(len(train_df), np.nan)
hybrid_oof = np.full(len(train_df), np.nan)
history_rows = []

for fold in sorted(train_df["fold"].unique()):
    tr = train_df[train_df["fold"] != fold].copy()
    va = train_df[train_df["fold"] == fold].copy()
    print(
        f"FOLD {fold}: "
        f"train={len(tr)} pos={int(tr[LABEL_COL].sum())}; "
        f"val={len(va)} pos={int(va[LABEL_COL].sum())}"
    )
    image_model, image_history = train_image_model(tr)
    hybrid_model, hybrid_history = train_hybrid_model(tr)
    image_oof[va.index.values] = predict_image_model(image_model, va)
    hybrid_oof[va.index.values] = predict_hybrid_model(hybrid_model, va)
    torch.save(image_model.state_dict(), OUTPUT_DIR / f"image_only_fold{fold}.pt")
    torch.save(hybrid_model.state_dict(), OUTPUT_DIR / f"hybrid_fold{fold}.pt")
    image_history["fold"] = fold
    image_history["model"] = "image_only"
    hybrid_history["fold"] = fold
    hybrid_history["model"] = "hybrid"
    history_rows.extend([*image_history.to_dict("records"), *hybrid_history.to_dict("records")])
    print("saved fold checkpoints")

np.save(OUTPUT_DIR / "image_only_oof.npy", image_oof)
np.save(OUTPUT_DIR / "hybrid_oof.npy", hybrid_oof)
pd.DataFrame(history_rows).to_csv(OUTPUT_DIR / "training_history.csv", index=False)


## Метрики OOF

In [ ]:
y = train_df[LABEL_COL].values
scores = {
    "image_only": image_oof,
    "hybrid": hybrid_oof,
    "blend_0.7_image_0.3_hybrid": 0.7 * image_oof + 0.3 * hybrid_oof,
    "blend_0.5_image_0.5_hybrid": 0.5 * image_oof + 0.5 * hybrid_oof,
}


def metrics(score):
    return {"auc": roc_auc_score(y, score), "ap": average_precision_score(y, score)}


comparison = pd.DataFrame(
    [{"model": name, **metrics(score)} for name, score in scores.items()]
).sort_values(["auc", "ap"], ascending=False)
print(comparison.to_string(index=False))
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)


def cluster_bootstrap_auc(y, score, groups, n_boot=2000, seed=42):
    y = np.asarray(y).astype(int)
    score = np.asarray(score, dtype=float)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    rng = np.random.RandomState(seed)
    vals = []
    for _ in range(n_boot):
        picked = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([np.where(groups == group)[0] for group in picked])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], score[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return roc_auc_score(y, score), float(lo), float(hi)


bootstrap_df = pd.DataFrame(
    [
        {"model": name, "auc": auc, "auc_lo": lo, "auc_hi": hi}
        for name, score in scores.items()
        for auc, lo, hi in [cluster_bootstrap_auc(y, score, train_df["study_id"].values)]
    ]
).sort_values("auc", ascending=False)
print(bootstrap_df.to_string(index=False))
bootstrap_df.to_csv(OUTPUT_DIR / "bootstrap_auc.csv", index=False)

threshold_rows = []
for name, score in scores.items():
    ev = U.evaluate_oof(
        y,
        score.reshape(1, -1),
        train_df["fold"].values.reshape(1, -1),
        train_df["study_id"].values,
        name=name,
        verbose=False,
    )
    thr = float(ev["deployed_threshold"])
    pred = (score >= thr).astype(int)
    m = U.binary_metrics(y, pred)
    threshold_rows.append(
        {
            "model": name,
            "threshold": thr,
            "f1": m["f1"],
            "sens": m["sens"],
            "spec": m["spec"],
            "precision": m["precision"],
            "tp": m["tp"],
            "fp": m["fp"],
            "fn": m["fn"],
            "tn": m["tn"],
        }
    )
threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))
threshold_df.to_csv(OUTPUT_DIR / "thresholds_candidates.csv", index=False)


## Ошибки и манифест

Лучший кандидат по OOF AUC выбирается только для разбора ошибок. Это не независимый тест. В инференс идёт не он, а смесь `0.7 / 0.3` с порогом `0.193505`.


In [ ]:
best_model_name = comparison.iloc[0]["model"]
best_score = scores[best_model_name]
best_threshold = float(threshold_df.loc[threshold_df["model"] == best_model_name, "threshold"].iloc[0])
audit = train_df.copy()
audit["score"] = best_score
audit["pred"] = (best_score >= best_threshold).astype(int)
audit["error"] = np.where(
    (audit[LABEL_COL] == 0) & (audit["pred"] == 1),
    "FP",
    np.where((audit[LABEL_COL] == 1) & (audit["pred"] == 0), "FN", "correct"),
)
audit.to_csv(OUTPUT_DIR / "best_candidate_oof_audit.csv", index=False)
print("Best candidate by OOF AUC:", best_model_name)
print(audit["error"].value_counts())
print("Top FP:")
print(
    audit.loc[audit["error"] == "FP"]
    .sort_values("score", ascending=False)[["study_id", "femur_side", LABEL_COL, "score", "file_path"]]
    .head(15)
    .to_string(index=False)
)
print("Top FN:")
print(
    audit.loc[audit["error"] == "FN"]
    .sort_values("score")[["study_id", "femur_side", LABEL_COL, "score", "file_path"]]
    .head(15)
    .to_string(index=False)
)

manifest = {
    "selection_reference": "OOF AUC development comparison",
    "best_candidate_by_oof_auc": best_model_name,
    "best_candidate_threshold": best_threshold,
    "image_only_checkpoints": [str(path) for path in sorted(OUTPUT_DIR.glob("image_only_fold*.pt"))],
    "hybrid_checkpoints": [str(path) for path in sorted(OUTPUT_DIR.glob("hybrid_fold*.pt"))],
    "comparison_file": str(OUTPUT_DIR / "comparison.csv"),
    "thresholds_file": str(OUTPUT_DIR / "thresholds_candidates.csv"),
}
with open(OUTPUT_DIR / "FINAL_CANDIDATE_MANIFEST.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("production thresholds.json не изменён")

best = comparison.iloc[0]
print(f"Candidate with highest OOF AUC: {best['model']}")
print(f"AUC={best['auc']:.3f}, AP={best['ap']:.3f}")


## Уже посчитанный результат

Эти цифры с прошлого прогона. Чтобы их увидеть ещё раз, обучение запускать не нужно.

```
prosthesis rows excluded: 1 studies: 1
images=147, studies=76, positives=34, negatives=113
fold 0: 22/6   fold 1: 19/10   fold 2: 24/6   fold 3: 23/7   fold 4: 25/5
Cached images: 147

blend_0.5_image_0.5_hybrid  AUC 0.699  AP 0.442  threshold 0.226  sens 0.588  spec 0.796
blend_0.7_image_0.3_hybrid  AUC 0.693  AP 0.461  threshold 0.194  sens 0.618  spec 0.779
hybrid                      AUC 0.693  AP 0.425  threshold 0.220  sens 0.676  spec 0.673
image_only                  AUC 0.676  AP 0.455  threshold 0.176  sens 0.529  spec 0.788

Best candidate by OOF AUC: blend_0.5_image_0.5_hybrid
errors for that candidate: correct 110, FP 23, FN 14
```

В `inference.py` подключена строка `0.7 / 0.3`, порог `0.193505`.
